# Web Scraping Theory Notes

---

## 1️⃣ What is Web Scraping?

**Definition:**  
Web scraping is the process of **automatically extracting information from websites**.

**Purpose:**  
- Collect data for research, analysis, or automation.  
- Example use-cases: price comparison, job listings, news scraping, data collection for ML.

**Common Tools:**  
- Python libraries: `BeautifulSoup`, `requests`, `Selenium`, `Scrapy`  
- Browser developer tools (Inspect Element)

**How it works:**  
1. Send a request to a webpage.  
2. Retrieve the HTML content.  
3. Parse the HTML to extract specific data.  
4. Store the extracted data (CSV, JSON, database).

---

## 2️⃣ HTTP Requests

**Definition:**  
HTTP requests are messages sent from a client (like Python) to a server to get or send data.

### Types of Requests

| Method   | Purpose                       |
|----------|-------------------------------|
| **GET**    | Retrieve data from a server  |
| **POST**   | Send data to a server        |
| **PUT**    | Update existing data on a server |
| **DELETE** | Remove data from a server    |

### Example in Python

```python
import requests

# Send a GET request
response = requests.get("https://example.com")

# Check status code
print(response.status_code)

# Print page content
print(response.text)


## HTML Structure

**Definition:**  
HTML (HyperText Markup Language) is used to structure web pages. It defines the layout and content that browsers display.

### Basic HTML Tags

| Tag       | Purpose                                  |
|-----------|------------------------------------------|
| `<html>`  | Root of the page                          |
| `<head>`  | Contains meta information, title, scripts |
| `<body>`  | Visible content of the webpage            |
| `<p>`     | Paragraph                                 |
| `<a>`     | Hyperlink                                 |
| `<div>`   | Block container                           |
| `<span>`  | Inline container                          |
| `<img>`   | Image                                     |
| `<ul>` / `<ol>` | Lists                                 |
| `<li>`    | List item                                 |

### Nested Structure
- HTML elements can be **nested inside each other**.
- Example:

```html
<div class="container">
  <p>Welcome to <a href="https://example.com">Example</a></p>
</div>


## Pagination Logic

# Definition:
Pagination is the method websites use to split large amounts of data across multiple pages. In web scraping, pagination means navigating through all these pages to collect complete data.

# Types of Pagination
|Type                 |	Example               |	Description |
|-------------------- |------------------------|----------------------------------------------|
|**Next Page Button** |	/products?page=2       |	Follow the "Next" link until it no longer exists.|
|**Page Numbers**	  |?page=1, ?page=2, ?page=3|	Iterate through numeric page values.|
|**Offset Pagination**|	?start=0, ?start=20	    |Data fetched using index offsets.|

**Basic Pagination Flow**

- Load the first page

- Extract the data

- Check if a next page link exists

- If yes → go to next page

- If no → stop scraping


## CSS Selectors vs XPath

| Feature     | CSS Selector                 | XPath                         |
|------------|------------------------------|-------------------------------|
| Syntax      | `.class`, `#id`, `tag > tag` | `/html/body/div/p`            |
| Use         | Select elements by class/id  | Navigate XML/HTML tree        |
| Readability | Simple, front-end friendly   | More precise, can be complex  |
| Example     | `soup.select('p.title')`     | `tree.xpath('//p[@class="title"]')` |



## Status Codes & Error Handling

**HTTP Status Codes:** Indicate the result of a web request.

| Code  | Meaning                   |
|-------|---------------------------|
| 200   | OK (success)              |
| 301   | Redirect                  |
| 400   | Bad Request               |
| 401   | Unauthorized              |
| 403   | Forbidden                 |
| 404   | Not Found                 |
| 500   | Internal Server Error     |

### Python Example

```python
import requests

try:
    response = requests.get("https://example.com")
    response.raise_for_status()  # Raise exception for 4xx/5xx errors
    print("Request Successful!")
except requests.exceptions.HTTPError as err:
    print("HTTP error:", err)
except requests.exceptions.RequestException as e:
    print("Other error:", e)


## TASK 1

In [1]:
import requests
from bs4 import BeautifulSoup

# 1. Website URL
url = "https://books.toscrape.com/catalogue/page-1.html"

# 2. Send HTTP GET request
response = requests.get(url)

# 3. Parse HTML
soup = BeautifulSoup(response.text, "html.parser")

# 4. Select all book blocks
books = soup.select(".product_pod")

# 5. Extract title, price, rating
for book in books:
    # Title
    title = book.h3.a["title"]

    # Price
    price = book.select_one(".price_color").text

    # Rating (stored as class: e.g., "star-rating Three")
    rating = book.p["class"][1]

    print(f"Title: {title}")
    print(f"Price: {price}")
    print(f"Rating: {rating}")
    print("-" * 40)

Title: A Light in the Attic
Price: Â£51.77
Rating: Three
----------------------------------------
Title: Tipping the Velvet
Price: Â£53.74
Rating: One
----------------------------------------
Title: Soumission
Price: Â£50.10
Rating: One
----------------------------------------
Title: Sharp Objects
Price: Â£47.82
Rating: Four
----------------------------------------
Title: Sapiens: A Brief History of Humankind
Price: Â£54.23
Rating: Five
----------------------------------------
Title: The Requiem Red
Price: Â£22.65
Rating: One
----------------------------------------
Title: The Dirty Little Secrets of Getting Your Dream Job
Price: Â£33.34
Rating: Four
----------------------------------------
Title: The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull
Price: Â£17.93
Rating: Three
----------------------------------------
Title: The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics
Price: Â£22.60
Rating: Four
-----

## TASK 2

In [2]:
import requests
from bs4 import BeautifulSoup

base_url = "http://books.toscrape.com/catalogue/page-{}.html"

all_books = []   # store all results here

page = 1
while True:
    print(f"Scraping Page {page}...")

    url = base_url.format(page)
    response = requests.get(url)

    # if page doesn't exist (404), stop the loop
    if response.status_code == 404:
        print("No more pages. Stopping.")
        break

    soup = BeautifulSoup(response.text, "html.parser")

    # select all book containers
    books = soup.select(".product_pod")

    # if no books found, stop
    if not books:
        print("No books found on this page. Stopping.")
        break

    for book in books:
        # extract title
        title = book.h3.a["title"]

        # extract price
        price = book.find("p", class_="price_color").text

        # extract rating
        rating_tag = book.find("p", class_="star-rating")
        rating = rating_tag["class"][1]   # second class is rating (e.g., 'Three')

        # store inside dictionary
        book_info = {
            "title": title,
            "price": price,
            "rating": rating
        }

        all_books.append(book_info)

    page += 1   # move to next page

print("Scraping completed!")
print(f"Total books scraped: {len(all_books)}")


Scraping Page 1...
Scraping Page 2...
Scraping Page 3...
Scraping Page 4...
Scraping Page 5...
Scraping Page 6...
Scraping Page 7...
Scraping Page 8...
Scraping Page 9...
Scraping Page 10...
Scraping Page 11...
Scraping Page 12...
Scraping Page 13...
Scraping Page 14...
Scraping Page 15...
Scraping Page 16...
Scraping Page 17...
Scraping Page 18...
Scraping Page 19...
Scraping Page 20...
Scraping Page 21...
Scraping Page 22...
Scraping Page 23...
Scraping Page 24...
Scraping Page 25...
Scraping Page 26...
Scraping Page 27...
Scraping Page 28...
Scraping Page 29...
Scraping Page 30...
Scraping Page 31...
Scraping Page 32...
Scraping Page 33...
Scraping Page 34...
Scraping Page 35...
Scraping Page 36...
Scraping Page 37...
Scraping Page 38...
Scraping Page 39...
Scraping Page 40...
Scraping Page 41...
Scraping Page 42...
Scraping Page 43...
Scraping Page 44...
Scraping Page 45...
Scraping Page 46...
Scraping Page 47...
Scraping Page 48...
Scraping Page 49...
Scraping Page 50...
Scraping 

In [42]:
import requests
from bs4 import BeautifulSoup
url = "https://books.toscrape.com/catalogue/page-1.html"
response = requests.get(url)
html = response.text

soup = BeautifulSoup(html, "html.parser")
# print(soup.prettify())
# print(soup)
books = soup.find_all("article", class_="product_pod")
# print(books)
for book in books:
    # 5. Extract title, price, rating
    # title = book.a["title"]
    title =book.h3.a["title"]
    price =book.find("p", class_="price_color").text
    rating = book.p["class"][1]
    stock = book.find("p", class_="instock availability").text.strip()
    print(f"Stock:{stock}")
    print(f"Title: {title}")
    print(f"Price: {price}")
    print(f"Rating: {rating}")
    print("-" * 40)
    
    

Stock:In stock
Title: A Light in the Attic
Price: Â£51.77
Rating: Three
----------------------------------------
Stock:In stock
Title: Tipping the Velvet
Price: Â£53.74
Rating: One
----------------------------------------
Stock:In stock
Title: Soumission
Price: Â£50.10
Rating: One
----------------------------------------
Stock:In stock
Title: Sharp Objects
Price: Â£47.82
Rating: Four
----------------------------------------
Stock:In stock
Title: Sapiens: A Brief History of Humankind
Price: Â£54.23
Rating: Five
----------------------------------------
Stock:In stock
Title: The Requiem Red
Price: Â£22.65
Rating: One
----------------------------------------
Stock:In stock
Title: The Dirty Little Secrets of Getting Your Dream Job
Price: Â£33.34
Rating: Four
----------------------------------------
Stock:In stock
Title: The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull
Price: Â£17.93
Rating: Three
----------------------------------------
Stock:In stock


"It's Only the Himalayas"

'Â£45.17'

'In stock'